# Cubierta Forestal · experimentación con AutoGluon + MLflow

Entrena **20 modelos** (5 familias × 4 combinaciones de hiperparámetros) con AutoGluon sobre el dataset
versionado en la base de datos de negocio (s5 · `training.covertype_features`) y registra cada ejecución en MLflow (s2):

| Qué | Dónde queda guardado |
|---|---|
| Experimentos, runs, parámetros, métricas, tags, registro de modelos | s1 · PostgreSQL `mlflow-db` (backend store) |
| Binarios del modelo (predictor de AutoGluon), `MLmodel`, firma, ejemplo de entrada | s3 · MinIO bucket `mlflow` (artifact store, vía proxy del servidor) |

Al final, el mejor modelo según *accuracy* en validación recibe el alias `production`,
que es el que carga la API de inferencia (`models:/cubierta-forestal@production`).

> Cada ejecución completa del notebook crea 20 versiones nuevas del modelo registrado.

In [1]:
import logging
import os
import shutil
import tempfile
import time
import warnings

# Silenciar ruido conocido: sin git en el contenedor, sin torch/GPU (se entrena en CPU),
# FutureWarnings de pandas dentro de AutoGluon y logs/barras de progreso de MLflow.
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
os.environ["MLFLOW_LOGGING_LEVEL"] = "ERROR"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*integer column.*")
logging.getLogger("autogluon.common.utils.resource_utils").setLevel(logging.CRITICAL)

import mlflow
import pandas as pd
from autogluon.tabular import TabularPredictor
from autogluon.tabular import __version__ as autogluon_version
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score
from sqlalchemy import URL, create_engine, text

from inference_api.autogluon_model import PREDICTOR_ARTIFACT, AutoGluonModel

EXPERIMENT_NAME = "cubierta-forestal-autogluon"
MODEL_NAME = os.getenv("MODEL_NAME", "cubierta-forestal")
MODEL_ALIAS = "production"   # alias que resuelve la API de inferencia
DATASET_VERSION = None       # None = la versión más reciente de training.dataset_version
LABEL = "cover_type"
TIME_LIMIT_S = 300           # tope de entrenamiento por modelo

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow:", mlflow.get_tracking_uri(), "| experimento:", experiment.name, "| artifacts:", experiment.artifact_location)

MLflow: http://mlflow:5000 | experimento: cubierta-forestal-autogluon | artifacts: mlflow-artifacts:/1


## 1. Datos desde la base de datos de negocio (s5)

In [2]:
engine = create_engine(URL.create(
    "postgresql+psycopg",
    username=os.environ["DATA_DB_USER"],
    password=os.environ["DATA_DB_PASSWORD"],
    host=os.environ["DATA_DB_HOST"],
    port=int(os.environ["DATA_DB_PORT"]),
    database=os.environ["DATA_DB_NAME"],
))

FEATURES = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
    "wilderness_area", "soil_type",
]

with engine.connect() as conn:
    if DATASET_VERSION is None:
        DATASET_VERSION = conn.execute(text(
            "SELECT version FROM training.dataset_version ORDER BY created_at DESC LIMIT 1"
        )).scalar_one()
    data = pd.read_sql(
        text(f"SELECT split, {', '.join(FEATURES)}, {LABEL} FROM training.covertype_features "
             "WHERE dataset_version = :version"),
        conn,
        params={"version": DATASET_VERSION},
    )

splits = {name: df.drop(columns="split").reset_index(drop=True) for name, df in data.groupby("split")}
train_df, val_df, test_df = splits["train"], splits["validation"], splits["test"]
print(f"dataset_version={DATASET_VERSION} train={len(train_df)} validation={len(val_df)} test={len(test_df)}")
train_df.head()

dataset_version=1 train=32460 validation=6828 test=6805


,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,cover_type
0,3377,310,12,858,141,1045,187,232,185,2118,commanche,c7756,0
1,2940,87,14,741,142,3190,240,216,104,708,commanche,c4703,1
2,3372,158,12,962,170,3113,233,242,138,997,commanche,c8771,6
3,2879,42,20,342,29,1733,219,193,101,830,commanche,c7756,1
4,3447,58,11,0,0,3247,228,218,122,2489,rawah,c8703,6


## 2. Espacio de búsqueda: 20 configuraciones

Claves de familia de AutoGluon: `RF` (Random Forest), `XT` (Extra Trees), `KNN`, `GBM` (LightGBM), `CAT` (CatBoost).

In [3]:
SEARCH_SPACE = [
    ("RF",  {"n_estimators": 100, "max_features": "sqrt", "criterion": "gini"}),
    ("RF",  {"n_estimators": 200, "max_features": "sqrt", "criterion": "entropy"}),
    ("RF",  {"n_estimators": 150, "max_features": 0.5,    "criterion": "gini", "min_samples_leaf": 2}),
    ("RF",  {"n_estimators": 100, "max_features": "log2", "criterion": "gini", "max_depth": 20}),
    ("XT",  {"n_estimators": 100, "max_features": "sqrt", "criterion": "gini"}),
    ("XT",  {"n_estimators": 200, "max_features": "sqrt", "criterion": "entropy"}),
    ("XT",  {"n_estimators": 150, "max_features": 0.5,    "criterion": "gini", "min_samples_leaf": 2}),
    ("XT",  {"n_estimators": 100, "max_features": "log2", "criterion": "gini", "max_depth": 20}),
    ("KNN", {"n_neighbors": 3,  "weights": "distance"}),
    ("KNN", {"n_neighbors": 7,  "weights": "distance"}),
    ("KNN", {"n_neighbors": 15, "weights": "uniform"}),
    ("KNN", {"n_neighbors": 31, "weights": "distance"}),
    ("GBM", {"num_boost_round": 300,  "learning_rate": 0.10, "num_leaves": 31}),
    ("GBM", {"num_boost_round": 600,  "learning_rate": 0.05, "num_leaves": 63}),
    ("GBM", {"num_boost_round": 1000, "learning_rate": 0.05, "num_leaves": 127, "feature_fraction": 0.8}),
    ("GBM", {"num_boost_round": 300,  "learning_rate": 0.20, "num_leaves": 255, "min_data_in_leaf": 10}),
    ("CAT", {"iterations": 500,  "learning_rate": 0.10, "depth": 6}),
    ("CAT", {"iterations": 1000, "learning_rate": 0.05, "depth": 8}),
    ("CAT", {"iterations": 800,  "learning_rate": 0.10, "depth": 10, "l2_leaf_reg": 5}),
    ("CAT", {"iterations": 1500, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 1}),
]
assert len(SEARCH_SPACE) == 20

## 3. Entrenamiento: una ejecución de MLflow y una versión del modelo registrado por configuración

In [4]:
PIP_REQUIREMENTS = [
    f"autogluon.tabular[catboost,lightgbm]=={autogluon_version}",
    f"mlflow=={mlflow.__version__}",
    f"pandas=={pd.__version__}",
]
X_val, y_val = val_df.drop(columns=LABEL), val_df[LABEL]
X_test, y_test = test_df.drop(columns=LABEL), test_df[LABEL]
train_dataset = mlflow.data.from_pandas(train_df, name=f"covertype-v{DATASET_VERSION}-train", targets=LABEL)
val_dataset = mlflow.data.from_pandas(val_df, name=f"covertype-v{DATASET_VERSION}-validation", targets=LABEL)


def run_experiment(idx: int, family: str, hyperparameters: dict) -> dict:
    predictor_dir = tempfile.mkdtemp(prefix=f"ag-{idx:02d}-{family}-")
    with mlflow.start_run(run_name=f"{idx:02d}-{family}") as run:
        mlflow.set_tags({"framework": "autogluon", "model_family": family})
        mlflow.log_params({
            "model_family": family,
            "dataset_version": DATASET_VERSION,
            "time_limit_s": TIME_LIMIT_S,
            **{f"hp_{k}": v for k, v in hyperparameters.items()},
        })
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(val_dataset, context="validation")

        start = time.perf_counter()
        predictor = TabularPredictor(
            label=LABEL, eval_metric="accuracy", path=predictor_dir, verbosity=0,
        ).fit(
            train_data=train_df,
            tuning_data=val_df,
            hyperparameters={family: hyperparameters},
            fit_weighted_ensemble=False,
            time_limit=TIME_LIMIT_S,
            num_gpus=0,
            save_space=True,
        )
        fit_time = time.perf_counter() - start

        val_pred, test_pred = predictor.predict(X_val), predictor.predict(X_test)
        metrics = {
            "val_accuracy": accuracy_score(y_val, val_pred),
            "val_f1_macro": f1_score(y_val, val_pred, average="macro"),
            "test_accuracy": accuracy_score(y_test, test_pred),
            "test_f1_macro": f1_score(y_test, test_pred, average="macro"),
            "fit_time_s": fit_time,
        }

        # Binarios + MLmodel -> MinIO; registro de la versión -> mlflow-db
        model_info = mlflow.pyfunc.log_model(
            name="model",
            python_model=AutoGluonModel(),
            artifacts={PREDICTOR_ARTIFACT: predictor_dir},
            signature=infer_signature(X_val, val_pred),
            input_example=X_val.head(5),
            pip_requirements=PIP_REQUIREMENTS,
            registered_model_name=MODEL_NAME,
        )
        # Ligadas al run y al LoggedModel (visibles en la pestaña Models de MLflow 3)
        mlflow.log_metrics(metrics, model_id=model_info.model_id)

    shutil.rmtree(predictor_dir, ignore_errors=True)
    return {"run": run.info.run_name, "run_id": run.info.run_id,
            "version": model_info.registered_model_version, **metrics}


results = []
for idx, (family, hyperparameters) in enumerate(SEARCH_SPACE, start=1):
    result = run_experiment(idx, family, hyperparameters)
    results.append(result)
    print(f"{result['run']:>8}  v{result['version']:<3} val_acc={result['val_accuracy']:.4f} "
          f"test_acc={result['test_accuracy']:.4f} fit={result['fit_time_s']:.1f}s")

Registered model 'cubierta-forestal' already exists. Creating a new version of this model...
Created version '21' of model 'cubierta-forestal'.


🏃 View run 01-RF at: http://mlflow:5000/#/experiments/1/runs/6fa5d03ca3c2440ba320ec47ffb1248a
🧪 View experiment at: http://mlflow:5000/#/experiments/1
   01-RF  v21  val_acc=0.9628 test_acc=0.9612 fit=1.2s


Registered model 'cubierta-forestal' already exists. Creating a new version of this model...
Created version '22' of model 'cubierta-forestal'.


🏃 View run 02-RF at: http://mlflow:5000/#/experiments/1/runs/8534f7d839f3445591e29e2855785303
🧪 View experiment at: http://mlflow:5000/#/experiments/1
   02-RF  v22  val_acc=0.9641 test_acc=0.9633 fit=1.4s


Registered model 'cubierta-forestal' already exists. Creating a new version of this model...
Created version '23' of model 'cubierta-forestal'.


...

🏃 View run 18-CAT at: http://mlflow:5000/#/experiments/1/runs/dad7da059c4d4efcab074796766fd3c8
🧪 View experiment at: http://mlflow:5000/#/experiments/1
  18-CAT  v38  val_acc=0.9588 test_acc=0.9572 fit=26.3s


Registered model 'cubierta-forestal' already exists. Creating a new version of this model...
Created version '39' of model 'cubierta-forestal'.


🏃 View run 19-CAT at: http://mlflow:5000/#/experiments/1/runs/a9241a652d894956b4ddca60910a7083
🧪 View experiment at: http://mlflow:5000/#/experiments/1
  19-CAT  v39  val_acc=0.9676 test_acc=0.9641 fit=37.9s


Registered model 'cubierta-forestal' already exists. Creating a new version of this model...
Created version '40' of model 'cubierta-forestal'.


🏃 View run 20-CAT at: http://mlflow:5000/#/experiments/1/runs/9bbb853313c44099bac6068660828158
🧪 View experiment at: http://mlflow:5000/#/experiments/1
  20-CAT  v40  val_acc=0.9508 test_acc=0.9445 fit=35.6s


## 4. Resultados

In [5]:
results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
results_df

,run,run_id,version,val_accuracy,val_f1_macro,test_accuracy,test_f1_macro,fit_time_s
0,16-GBM,005c6e43499b41e5b7ce13ac62f6ac6a,36,0.969977,0.941868,0.965907,0.925168,4.215865
1,15-GBM,a418e90c462445df9c841c875e026588,35,0.968512,0.938362,0.963556,0.921906,4.213632
2,14-GBM,bdccc0fd4481494396c262c0d0fe4d4a,34,0.967926,0.941322,0.966054,0.929847,5.504451
3,13-GBM,73a4e55c98d24597ba8f618a923a6d99,33,0.967780,0.942945,0.965026,0.931836,3.811817
4,19-CAT,a9241a652d894956b4ddca60910a7083,39,0.967633,0.942167,0.964144,0.931676,37.871676
5,09-KNN,2e057de4abe745ceb48903f7d8114008,29,0.965876,0.942206,0.962381,0.926395,0.206544
6,02-RF,8534f7d839f3445591e29e2855785303,22,0.964118,0.933451,0.963262,0.920619,1.432135
7,10-KNN,642bf88f64ca48a585da4b8239248c98,30,0.963825,0.934665,0.964144,0.931297,0.225855
8,03-RF,4068581708a348b584d47b86880b6577,23,0.963679,0.928039,0.960911,0.912760,1.453687
9,01-RF,6fa5d03ca3c2440ba320ec47ffb1248a,21,0.962800,0.929372,0.961205,0.915494,1.156341


## 5. Promoción: alias `production` al mejor modelo según validación

In [6]:
best = results_df.iloc[0]
client = MlflowClient()
client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, best["version"])
client.set_model_version_tag(MODEL_NAME, best["version"], "selected_by", "val_accuracy")
print(f"{MODEL_NAME}@{MODEL_ALIAS} -> versión {best['version']} ({best['run']}), "
      f"val_acc={best['val_accuracy']:.4f}, test_acc={best['test_accuracy']:.4f}")

cubierta-forestal@production -> versión 36 (16-GBM), val_acc=0.9700, test_acc=0.9659


## 6. Verificación: cargar el modelo desde el registro como lo hará la API

In [7]:
production_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
sample = X_test.head(10)
pd.DataFrame({"predicho": production_model.predict(sample), "real": y_test.head(10)})

,predicho,real
0,2,2
1,0,0
2,1,1
3,6,6
4,2,2
5,0,0
6,0,0
7,0,0
8,2,2
9,0,0
